# GEN3D — Step 1: Data pipeline (Colab)
Download → check → preprocess **8,000 Objaverse-LVIS objects**. Results are saved to Google Drive.

**Before the first run:** upload `GEN3D_code.zip` to `MyDrive/GEN3D/`.

**If Colab disconnects:** just *Runtime → Run all* again. Every step resumes where it stopped
(already processed objects are never downloaded or processed again).

| Where | What | Size |
|---|---|---|
| Colab disk `/content/GEN/data/raw` (temporary) | downloaded GLB files | up to ~28 GB |
| Drive `MyDrive/GEN3D/data/meta` | CSV logs (candidates, checked, preprocessed) | small |
| Drive `MyDrive/GEN3D/data/processed` | training data (`data.npz` + `norm.json` per object) | ~7 GB |

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/GEN3D'
import os; os.makedirs(DRIVE, exist_ok=True)
print(os.listdir(DRIVE))

## 2. Unpack code + install libraries

In [ ]:
%%bash
set -e
rm -rf /content/GEN && mkdir -p /content/GEN
unzip -q /content/drive/MyDrive/GEN3D/GEN3D_code.zip -d /content/GEN
pip install -q objaverse trimesh pyyaml tqdm pandas scipy scikit-image pillow matplotlib
echo "CPU cores: $(nproc)"; df -h /content | tail -1

## 3. Keep meta + processed data on Drive (persistent), raw GLBs on Colab disk

In [ ]:
%%bash
set -e
D=/content/drive/MyDrive/GEN3D/data
mkdir -p $D/meta $D/processed /content/GEN/data/raw
# first run only: put our selection (candidates.csv) on Drive
[ -f $D/meta/candidates.csv ] || cp /content/GEN/data_seed/candidates.csv $D/meta/
rm -rf /content/GEN/data/meta /content/GEN/data/processed
ln -s $D/meta /content/GEN/data/meta
ln -s $D/processed /content/GEN/data/processed
ls -la /content/GEN/data

## 4. Download (resumable, ~10–30 min)

In [ ]:
%cd /content/GEN
!python scripts/download_objaverse.py --workers 16 2>&1 | grep -v "it/s]"

## 5. Check meshes (incremental)

In [ ]:
!python scripts/check_meshes.py 2>&1 | grep -v "it/s]"

## 6. Preprocess (resumable, the long step: ~3 s/object/core)
Log is saved every 100 objects. If the session dies, re-run all cells.

In [ ]:
import os
!python scripts/preprocess.py --workers {os.cpu_count()} 2>&1 | grep -v "it/s]"

## 7. Summary + visual check

In [ ]:
import pandas as pd
m = '/content/GEN/data/meta/'
cand = pd.read_csv(m + 'candidates.csv', keep_default_na=False)
chk = pd.read_csv(m + 'checked.csv', keep_default_na=False)
pre = pd.read_csv(m + 'preprocessed.csv', keep_default_na=False)
sel = cand[cand.selected.astype(str) == 'True']
ok = pre[(pre.status == 'ok') & (pre.flag == '')].merge(sel[['uid', 'group', 'category', 'split']], on='uid')
print(f"selected {len(sel)} | checked PASS {(chk.status == 'PASS').sum()} | preprocessed ok {(pre.status == 'ok').sum()} | clean {len(ok)}")
print('flags:', pre[pre.flag != ''].flag.value_counts().to_dict())
print(ok.groupby('group').agg(objects=('uid', 'size'), categories=('category', 'nunique')))
print(ok.split.value_counts())

In [ ]:
!python scripts/visualize_sample.py --n 8
from IPython.display import Image; Image('/content/GEN/outputs/preprocess_check.png')